In [1]:
import pandas as pd
from pandas.tseries.offsets import DateOffset
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import griddata
import datetime as dt
from pathlib import Path
import os
from tqdm import tqdm

In [2]:
# Find nearest index
def find_index(array, x):
    if array.ndim == 1:
        idx = np.argmin(np.abs(array - x))
    elif array.ndim == 2:
        idx = np.unravel_index(np.argmin(np.abs(array - x)), array.shape)
    else:
        raise ValueError("Unsupported array dimensions for find_index function.")
    return idx

In [3]:
#Read in SO4, NO3, NH4 wet dep timeseries files
daily_result_dir = Path('/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/Timeseries_atm_h2')
daily_so4_files_to_open = ['FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.WD_HNO3.nc', 
                           'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.WD_NH4.nc', 
                           'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.WD_NH3.nc',
                           'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.so4_a1SFWET.nc',
                           'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.so4_a2SFWET.nc',
                           'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.so4_a3SFWET.nc',
                           'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.so4_c1SFWET.nc',
                           'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.so4_c2SFWET.nc',
                           'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.so4_c3SFWET.nc']
                           
# Concatenate directory path with each file name separately
file_paths = [daily_result_dir / file_name for file_name in daily_so4_files_to_open]

# Open multiple netCDF files as a single dataset
nc_daily = xr.open_mfdataset(file_paths, combine='nested')
nc_daily 

<xarray.Dataset>
Dimensions:      (time: 7671, lat: 192, lon: 288)
Coordinates:
  * lat          (lat) float64 -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon          (lon) float64 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.2 357.5 358.8
  * time         (time) datetime64[ns] 2002-01-01 2002-01-02 ... 2023-01-01
Data variables:
    WD_HNO3      (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    WD_NH4       (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    WD_NH3       (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    so4_a1SFWET  (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    so4_a2SFWET  (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    so4_a3SFWET  (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    so4_c1SFWET  (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    so4_c2SFWET  (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    so4_c3SFWET  (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
Attributes:
    Conventions:       CF-1.0
    source:            CAM
    case:              FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001
    logname:           demurray
    host:              derecho8
    initial_file:      /glade/campaign/acom/acom-climate/UTLS/shawnh/archive/...
    topography_file:   /glade/campaign/cesm/cesmdata/inputdata/atm/cam/topo/f...
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    time_period_freq:  day_1

In [4]:
# Calculate SO4_wet_sum
nc_daily['SO4_wet_sum'] = (nc_daily['so4_a1SFWET'] + nc_daily['so4_a2SFWET'] + nc_daily['so4_a3SFWET'] 
                                + nc_daily['so4_c1SFWET'] + nc_daily['so4_c2SFWET'] + nc_daily['so4_c3SFWET']) * -1

# Calculate NO3_wet
nc_daily['NO3_wet'] = nc_daily['WD_HNO3'] * -1 * 0.984006

# Calculate NH4_wet
nc_daily['WD_NH4_corr'] = xr.where(nc_daily['WD_NH4'] > 0, 0, nc_daily['WD_NH4'])

# Calculate NH3_wet
nc_daily['WD_NH3_corr'] = xr.where(nc_daily['WD_NH3'] > 0, 0, nc_daily['WD_NH3'])

# Perform unit conversions
conversions = 86400 * 1000000  # seconds to days and kg to mg per m2
nc_daily = nc_daily.apply(lambda x: x * conversions if x.name in ['SO4_wet_sum', 'WD_NH4_corr', 'WD_NH3_corr', 'NO3_wet'] else x)
nc_daily  

<xarray.Dataset>
Dimensions:      (lat: 192, lon: 288, time: 7671)
Coordinates:
  * lat          (lat) float64 -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon          (lon) float64 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.2 357.5 358.8
  * time         (time) datetime64[ns] 2002-01-01 2002-01-02 ... 2023-01-01
Data variables: (12/13)
    WD_HNO3      (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    WD_NH4       (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    WD_NH3       (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    so4_a1SFWET  (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    so4_a2SFWET  (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    so4_a3SFWET  (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    ...           ...
    so4_c2SFWET  (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    so4_c3SFWET  (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    SO4_wet_sum  (time, lat, lon) float64 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    NO3_wet      (time, lat, lon) float64 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    WD_NH4_corr  (time, lat, lon) float64 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    WD_NH3_corr  (time, lat, lon) float64 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>

In [5]:
#Select cells that correspond to NADP sites: read in NADP lat/long and apply the find nearest function
pathData = '/glade/u/home/demurray/External File Uploads/NADP NTN wet dep data/'
os.chdir(pathData)
ntn = pd.read_csv('ntn.csv')

# Only select the cells in .nc that correspond to an NADP site lat/long
subset_list = []
progress_bar = tqdm(total=len(ntn))
for index, row in ntn.iterrows():
    lat = row['latitude']
    lon = 360-(row['longitude']*-1)   # longitude is positive and based on 360 degrees.
    lat_idx = find_index(nc_daily['lat'].values, lat)   # right now we are doing a 'find nearest' calculation, should probably interpolate across grid cell and have exact coordinates represented?
    lon_idx = find_index(nc_daily['lon'].values, lon)
    subset = nc_daily.isel(lat=lat_idx, lon=lon_idx)
    subset['siteId'] = row['siteId']  # Add 'siteId' as a new coordinate/index
    subset = subset.assign_coords(siteId=row['siteId'])
    subset_list.append(subset)
    progress_bar.update(1)
progress_bar.close()

# Concatenate the list of subsets into a new xarray dataset
nc_daily_nadp = xr.concat(subset_list, dim='siteId')
nc_daily_nadp

100%|██████████| 391/391 [00:01<00:00, 244.89it/s]


<xarray.Dataset>
Dimensions:      (siteId: 391, time: 7671)
Coordinates:
    lat          (siteId) float64 57.02 56.07 57.02 65.5 ... 42.88 39.11 40.99
    lon          (siteId) float64 248.8 248.8 248.8 212.5 ... 271.2 280.0 253.8
  * time         (time) datetime64[ns] 2002-01-01 2002-01-02 ... 2023-01-01
  * siteId       (siteId) <U4 'AB32' 'AB34' 'AB36' ... 'WI99' 'WV99' 'WY96'
Data variables: (12/13)
    WD_HNO3      (siteId, time) float32 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    WD_NH4       (siteId, time) float32 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    WD_NH3       (siteId, time) float32 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    so4_a1SFWET  (siteId, time) float32 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    so4_a2SFWET  (siteId, time) float32 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    so4_a3SFWET  (siteId, time) float32 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    ...           ...
    so4_c2SFWET  (siteId, time) float32 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    so4_c3SFWET  (siteId, time) float32 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    SO4_wet_sum  (siteId, time) float64 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    NO3_wet      (siteId, time) float64 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    WD_NH4_corr  (siteId, time) float64 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    WD_NH3_corr  (siteId, time) float64 dask.array<chunksize=(1, 7671), meta=np.ndarray>

In [14]:
''' NOTE THAT TIME STAMP HAS NOT BEEN CHANGED IN THESE FILES.
#Add a loop that for each siteId it writes a file to timeseries
output_directory = '/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/Daily_Timeseries_NADPsites'

unique_site_ids = nc_daily_nadp['siteId'].values

# Define the index threshold
index_threshold = 179 #doing this because had to pause file writing here.

# Find the indices of unique siteIds
indices = np.arange(len(unique_site_ids))

# Filter unique siteIds based on index
filtered_site_ids = unique_site_ids[indices >= index_threshold]

# Initialize tqdm
pbar = tqdm(filtered_site_ids, desc="Writing subset files")

# Iterate over each unique siteId
for site_id in pbar:
    # Subset the dataset for the current siteId
    subset_ds = nc_daily_nadp.where(nc_daily_nadp['siteId'] == site_id, drop=True)
    
    # Construct the filename
    filename = f'{site_id}_DailyTimeseries_WetDep_SO4NO3NH4.nc'
    
    # Write the subsetted dataset to the specified directory
    output_path = os.path.join(output_directory, filename)
    subset_ds.to_netcdf(output_path)
    
    # Update tqdm description
    pbar.set_description(f"Writing subset files: {filename}")
'''

Writing subset files: WY96_DailyTimeseries_WetDep_SO4NO3NH4.nc: 100%|██████████| 212/212 [5:17:10<00:00, 89.76s/it]   


In [ ]:
'''
#Read in a subset of daily files to make sure that it worked
daily_test_dir = Path('/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/Daily_Timeseries_NADPsites')
daily_test_to_open = ['WY98_DailyTimeseries_WetDep_SO4NO3NH4.nc', 'MS30_DailyTimeseries_WetDep_SO4NO3NH4.nc',
                      'WY99_DailyTimeseries_WetDep_SO4NO3NH4.nc']

# Concatenate directory path with each file name separately
file_paths = [daily_test_dir / file_name for file_name in daily_test_to_open]

# Open multiple netCDF files as a single dataset
nc_daily_test = xr.open_mfdataset(file_paths, combine='nested')
nc_daily_test
'''

In [6]:
#Turn nc_daily_nadp xarray into a pandas dataframe with similar attributes to the NADP dataset
mod_nadp = nc_daily_nadp.to_dataframe().reset_index()
mod_nadp = mod_nadp.replace(-0.0, 0)
mod_nadp = mod_nadp.rename(columns = {'time': 'model_time', 'SO4_wet_sum': 'Mod_SO4_mgm2', 
                                      'WD_NH4_corr': 'Mod_NH4_mgm2', 'NO3_wet':'Mod_NO3_mgm2', 'WD_NH3_corr': 'Mod_NH3_mgm2'})

# change the unit conversion of Mod_NH4_mgm2 because it didn't work in above code?
mod_nadp['Mod_NH4_mgm2'] = mod_nadp['Mod_NH4_mgm2'] * -1 #negative to positive, for some reason this did not work in above conversion code
mod_nadp['Mod_NH4_mgm2'] = mod_nadp['Mod_NH4_mgm2'].replace(-0.0, 0.0)

mod_nadp['Mod_NH3_mgm2'] = mod_nadp['Mod_NH3_mgm2'] * -1 #negative to positive, for some reason this did not work in above conversion code
mod_nadp['Mod_NH3_mgm2'] = mod_nadp['Mod_NH3_mgm2'].replace(-0.0, 0.0)

mod_nadp['Mod_NH4_NH3_mgm2'] = mod_nadp['Mod_NH4_mgm2'] + mod_nadp['Mod_NH3_mgm2']

#IMPORTANT STEP: change the time to one day prior (model writes time at end of current day)
mod_nadp['time'] = pd.to_datetime(mod_nadp['model_time'],format='%Y-%m-%d')+DateOffset(days=-1)

mod_nadp.head(10)

,siteId,model_time,lat,lon,WD_HNO3,WD_NH4,WD_NH3,so4_a1SFWET,so4_a2SFWET,so4_a3SFWET,so4_c1SFWET,so4_c2SFWET,so4_c3SFWET,Mod_SO4_mgm2,Mod_NO3_mgm2,Mod_NH4_mgm2,Mod_NH3_mgm2,Mod_NH4_NH3_mgm2,time
0,AB32,2002-01-01,57.015707,248.75,-1.395923e-25,1.784353e-27,-9.164005e-31,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,1.186787e-14,0.000000,7.917701e-20,7.917701e-20,2001-12-31
1,AB32,2002-01-02,57.015707,248.75,-1.970314e-15,-2.079935e-17,-1.916300e-17,0.000000e+00,0.000000e+00,0.000000e+00,-1.651060e-15,-1.473558e-17,-1.117747e-18,0.000144,1.675124e-04,0.000002,1.655683e-06,3.452747e-06,2002-01-01
2,AB32,2002-01-03,57.015707,248.75,-7.023518e-13,-4.363956e-14,-1.561877e-14,-2.656719e-16,-4.143680e-17,-1.641524e-15,-3.057411e-13,-1.495660e-13,-7.769978e-14,0.046220,5.971262e-02,0.003770,1.349461e-03,5.119919e-03,2002-01-02
3,AB32,2002-01-04,57.015707,248.75,-7.138383e-12,-1.705666e-13,-6.441399e-13,-4.020032e-16,-5.714549e-17,-3.779064e-15,-1.318216e-12,-1.442421e-13,-1.397894e-14,0.127930,6.068919e-01,0.014737,5.565369e-02,7.039064e-02,2002-01-03
4,AB32,2002-01-05,57.015707,248.75,-1.088934e-11,-1.722836e-12,-3.988795e-13,-7.043801e-16,-1.617886e-17,-1.412236e-15,-1.701545e-12,-1.126615e-13,-2.767494e-15,0.157171,9.257913e-01,0.148853,3.446319e-02,1.833162e-01,2002-01-04
5,AB32,2002-01-06,57.015707,248.75,-7.356965e-14,-5.876697e-15,-2.496924e-16,-4.431961e-19,-1.291710e-19,-1.136456e-17,-7.346153e-14,-1.230468e-15,-4.861741e-17,0.006459,6.254753e-03,0.000508,2.157342e-05,5.293200e-04,2002-01-05
6,AB32,2002-01-07,57.015707,248.75,-4.545261e-13,-9.093178e-15,-7.107025e-14,-4.109256e-17,-6.697324e-18,-5.986324e-16,-1.681824e-14,-1.763466e-15,-9.849780e-16,0.001746,3.864295e-02,0.000786,6.140470e-03,6.926120e-03,2002-01-06
7,AB32,2002-01-08,57.015707,248.75,-1.315498e-12,-1.628972e-14,-3.567871e-14,-1.229326e-16,-3.000804e-17,-1.774741e-15,-3.836262e-13,-5.532273e-14,-4.647567e-15,0.038493,1.118412e-01,0.001407,3.082640e-03,4.490072e-03,2002-01-07
8,AB32,2002-01-09,57.015707,248.75,-2.320464e-13,-6.295724e-15,-7.901189e-15,-1.366566e-17,-3.817968e-18,-2.509400e-16,-8.347699e-15,-3.687507e-15,-2.531511e-16,0.001085,1.972815e-02,0.000544,6.826628e-04,1.226613e-03,2002-01-08
9,AB32,2002-01-10,57.015707,248.75,-4.922058e-15,-3.858655e-17,-9.167616e-17,-3.179468e-19,-1.419141e-19,-8.516404e-18,-4.706183e-16,-6.118856e-16,-1.252620e-17,0.000095,4.184641e-04,0.000003,7.920820e-06,1.125470e-05,2002-01-09


In [7]:
#IGNORE THIS CELL IF RUNNING FULL ANALYSES, THIS IS JUST FOR COMPARISON WITH MUSICAV0 RUN
#truncate time
start = dt.datetime.strptime('2016-12-31', '%Y-%m-%d')
end = dt.datetime.strptime('2018-12-31', '%Y-%m-%d')
mod_nadp_sub = mod_nadp.loc[(mod_nadp.time > start) & (mod_nadp.time < end),]

#truncate variables
mod_nadp_sub = mod_nadp_sub[['siteId', 'model_time', 'lat', 'lon', 'Mod_SO4_mgm2', 'Mod_NO3_mgm2', 'Mod_NH4_mgm2', 'Mod_NH4_NH3_mgm2','time']]

#write file
mod_nadp_sub.to_csv('/glade/u/home/demurray/Murray-NCAR-GVP/MUSICA Analyses/Data outputs/SO4_NO3_NH4_global20172018_cleaned_nadpsitemerged.csv')

In [3]:
#read in timeseries of: NADP NTN and ensure correct/consistent formatting

# subset for NTN comparisons: SO4, NH4, NO3 (downloaded NADP NTN 2002-2023 for all sites at weekly scale on 3/25/2024 excluding invalids)
pathData = '/glade/u/home/demurray/External File Uploads/NADP NTN wet dep data/'
os.chdir(pathData)
nadp_df = pd.read_csv('NTN-ALL-Weekly 2000_2022.csv', parse_dates = ['dateOn', 'dateOff'])
nadp_df = nadp_df[['siteId', 'dateOn', 'dateOff','NH4', 'NO3', 'SO4', 'subppt']]
ntn = pd.read_csv('ntn.csv')

#replace all negative numbers with np.nan and assign units, perform conversions
nadp_df[nadp_df.select_dtypes(include='number') < 0] = np.nan
nadp_df = nadp_df.rename(columns = {'SO4':'SO4_mgL', 'NH4':'NH4_mgL', 'NO3':'NO3_mgL', 'subppt':'ppt_mm', }) #assign units

#Scale concentrations to mg/m2 using precip volume, assuming 1mm rain = 1L/m2
nadp_df['SO4_mgm2'] = (nadp_df['SO4_mgL'] * nadp_df['ppt_mm']) 
nadp_df['NH4_mgm2'] = (nadp_df['NH4_mgL'] * nadp_df['ppt_mm'])
nadp_df['NO3_mgm2'] = (nadp_df['NO3_mgL'] * nadp_df['ppt_mm'])

#Ensure correct datetime formatting
nadp_df['dateOn'] = pd.to_datetime(nadp_df['dateOn'], format='%m/%d/%Y %H:%M')
nadp_df['dateOff'] = pd.to_datetime(nadp_df['dateOff'], format='%m/%d/%Y %H:%M')
nadp_df = nadp_df.sort_values(['siteId', 'dateOn'], ascending = True)

#Need to round date because using DAILY data for modelled comparisons
nadp_df['dateOnround'] = nadp_df.dateOn + dt.timedelta(hours=12)
nadp_df['dateOnround'] = pd.to_datetime(nadp_df.dateOnround.dt.strftime('%Y-%m-%d'))
nadp_df['dateOffround'] = nadp_df.dateOff + dt.timedelta(hours=12)
nadp_df['dateOffround'] = pd.to_datetime(nadp_df.dateOffround.dt.strftime('%Y-%m-%d'))

#merge with site info and then select relevant columns
nadp_df = pd.merge(nadp_df, ntn, on = 'siteId')
nadp_df = nadp_df[['siteId','latitude', 'longitude', 'dateOn', 'dateOff', 'dateOnround', 'dateOffround', 'SO4_mgm2', 'NO3_mgm2', 'NH4_mgm2','ppt_mm']]

nadp_df.head(5)

,siteId,latitude,longitude,dateOn,dateOff,dateOnround,dateOffround,SO4_mgm2,NO3_mgm2,NH4_mgm2,ppt_mm
0,AB32,57.1894,-111.6406,2016-09-13 18:40:00,2016-09-20 15:10:00,2016-09-14,2016-09-21,NaN,NaN,NaN,0.762
1,AB32,57.1894,-111.6406,2016-09-20 15:15:00,2016-09-28 16:00:00,2016-09-21,2016-09-29,NaN,NaN,NaN,0.508
2,AB32,57.1894,-111.6406,2016-09-28 16:00:00,2016-10-05 16:55:00,2016-09-29,2016-10-06,6.888988,3.42646,0.703326,18.034
3,AB32,57.1894,-111.6406,2016-10-05 16:55:00,2016-10-11 17:00:00,2016-10-06,2016-10-12,NaN,NaN,NaN,10.160
4,AB32,57.1894,-111.6406,2016-10-11 17:00:00,2016-10-18 20:00:00,2016-10-12,2016-10-19,NaN,NaN,NaN,13.208


In [4]:
##Assign sampling intervals to NADP NTN deposition data
nadp_df['SamplingInt'] = pd.Series(dtype='int')
nadp_df['IntTime'] = pd.Series(dtype='int')

sites = nadp_df.siteId.unique()

for i in tqdm(sites, unit = 'sites', total = len(sites), ncols = 100):
    nadp_df.loc[nadp_df.siteId == i,'SamplingInt'] = list(range(0, len(nadp_df.loc[nadp_df.siteId == i]), 1))
    nadp_df.loc[nadp_df.siteId == i, 'IntTime'] = nadp_df.loc[nadp_df.siteId == i, 'dateOff'] - nadp_df.loc[nadp_df.siteId == i, 'dateOn']
nadp_df.head(50)

100%|██████████████████████████████████████████████████████████| 339/339 [00:34<00:00,  9.74sites/s]


,siteId,latitude,longitude,dateOn,dateOff,dateOnround,dateOffround,SO4_mgm2,NO3_mgm2,NH4_mgm2,ppt_mm,SamplingInt,IntTime
0,AB32,57.1894,-111.6406,2016-09-13 18:40:00,2016-09-20 15:10:00,2016-09-14,2016-09-21,NaN,NaN,NaN,0.762,0.0,6 days 20:30:00
1,AB32,57.1894,-111.6406,2016-09-20 15:15:00,2016-09-28 16:00:00,2016-09-21,2016-09-29,NaN,NaN,NaN,0.508,1.0,8 days 00:45:00
2,AB32,57.1894,-111.6406,2016-09-28 16:00:00,2016-10-05 16:55:00,2016-09-29,2016-10-06,6.888988,3.426460,0.703326,18.034,2.0,7 days 00:55:00
3,AB32,57.1894,-111.6406,2016-10-05 16:55:00,2016-10-11 17:00:00,2016-10-06,2016-10-12,NaN,NaN,NaN,10.160,3.0,6 days 00:05:00
4,AB32,57.1894,-111.6406,2016-10-11 17:00:00,2016-10-18 20:00:00,2016-10-12,2016-10-19,NaN,NaN,NaN,13.208,4.0,7 days 03:00:00
5,AB32,57.1894,-111.6406,2016-10-18 20:00:00,2016-10-25 18:00:00,2016-10-19,2016-10-26,3.940048,6.443472,1.219708,3.556,5.0,6 days 22:00:00
6,AB32,57.1894,-111.6406,2016-10-25 18:00:00,2016-10-31 16:47:00,2016-10-26,2016-11-01,6.391656,9.125712,3.959352,4.572,6.0,5 days 22:47:00
7,AB32,57.1894,-111.6406,2016-10-31 16:47:00,2016-11-07 17:00:00,2016-11-01,2016-11-08,0.890016,0.431292,0.190500,1.524,7.0,7 days 00:13:00
8,AB32,57.1894,-111.6406,2016-11-07 17:00:00,2016-11-15 16:00:00,2016-11-08,2016-11-16,0.141732,0.358902,0.068326,0.254,8.0,7 days 23:00:00
9,AB32,57.1894,-111.6406,2016-11-15 16:00:00,2016-11-22 18:30:00,2016-11-16,2016-11-23,0.672084,1.280160,0.083312,0.508,9.0,7 days 02:30:00


In [10]:
#Write loop to assign sampling intervals to  modelled data frame
mod_nadp['SamplingInt'] = pd.Series(dtype='int') # Add a SamplingInt column to the modelled
sites =nadp_df.siteId.unique()#[0:170]

for i in tqdm(sites, unit = "sites", total = len(sites), ncols = 100):
    sampleInt = nadp_df.loc[nadp_df.siteId==i,'SamplingInt']
    #print(i)
    for j in sampleInt:
       #print(j)
       begDate = pd.Timestamp(nadp_df.loc[(nadp_df.siteId == i) & (nadp_df.SamplingInt == j), 'dateOnround'].item())
       endDate = pd.Timestamp(nadp_df.loc[(nadp_df.siteId == i) & (nadp_df.SamplingInt == j), 'dateOffround'].item())
       #print(endDate)
       mod_nadp.loc[(mod_nadp.siteId == i) & (mod_nadp.time >= begDate) & (mod_nadp.time < endDate), 'SamplingInt'] = j 

mod_nadp.drop_duplicates(inplace = True)
mod_nadp.to_csv('/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/Daily_Timeseries_NADPsites/SO4_NO3_NH3_NH4_wetdep_Daily_SamplingIntAssigned_Allsites.csv')
mod_nadp.head(10)

100%|██████████████████████████████████████████████████████| 339/339 [17:14:19<00:00, 183.07s/sites]


,siteId,model_time,lat,lon,WD_HNO3,WD_NH4,WD_NH3,so4_a1SFWET,so4_a2SFWET,so4_a3SFWET,so4_c1SFWET,so4_c2SFWET,so4_c3SFWET,Mod_SO4_mgm2,Mod_NO3_mgm2,Mod_NH4_mgm2,Mod_NH3_mgm2,Mod_NH4_NH3_mgm2,time,SamplingInt
0,AB32,2002-01-01,57.015707,248.75,-1.395923e-25,1.784353e-27,-9.164005e-31,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,1.186787e-14,0.000000,7.917701e-20,7.917701e-20,2001-12-31,NaN
1,AB32,2002-01-02,57.015707,248.75,-1.970314e-15,-2.079935e-17,-1.916300e-17,0.000000e+00,0.000000e+00,0.000000e+00,-1.651060e-15,-1.473558e-17,-1.117747e-18,0.000144,1.675124e-04,0.000002,1.655683e-06,3.452747e-06,2002-01-01,NaN
2,AB32,2002-01-03,57.015707,248.75,-7.023518e-13,-4.363956e-14,-1.561877e-14,-2.656719e-16,-4.143680e-17,-1.641524e-15,-3.057411e-13,-1.495660e-13,-7.769978e-14,0.046220,5.971262e-02,0.003770,1.349461e-03,5.119919e-03,2002-01-02,NaN
3,AB32,2002-01-04,57.015707,248.75,-7.138383e-12,-1.705666e-13,-6.441399e-13,-4.020032e-16,-5.714549e-17,-3.779064e-15,-1.318216e-12,-1.442421e-13,-1.397894e-14,0.127930,6.068919e-01,0.014737,5.565369e-02,7.039064e-02,2002-01-03,NaN
4,AB32,2002-01-05,57.015707,248.75,-1.088934e-11,-1.722836e-12,-3.988795e-13,-7.043801e-16,-1.617886e-17,-1.412236e-15,-1.701545e-12,-1.126615e-13,-2.767494e-15,0.157171,9.257913e-01,0.148853,3.446319e-02,1.833162e-01,2002-01-04,NaN
5,AB32,2002-01-06,57.015707,248.75,-7.356965e-14,-5.876697e-15,-2.496924e-16,-4.431961e-19,-1.291710e-19,-1.136456e-17,-7.346153e-14,-1.230468e-15,-4.861741e-17,0.006459,6.254753e-03,0.000508,2.157342e-05,5.293200e-04,2002-01-05,NaN
6,AB32,2002-01-07,57.015707,248.75,-4.545261e-13,-9.093178e-15,-7.107025e-14,-4.109256e-17,-6.697324e-18,-5.986324e-16,-1.681824e-14,-1.763466e-15,-9.849780e-16,0.001746,3.864295e-02,0.000786,6.140470e-03,6.926120e-03,2002-01-06,NaN
7,AB32,2002-01-08,57.015707,248.75,-1.315498e-12,-1.628972e-14,-3.567871e-14,-1.229326e-16,-3.000804e-17,-1.774741e-15,-3.836262e-13,-5.532273e-14,-4.647567e-15,0.038493,1.118412e-01,0.001407,3.082640e-03,4.490072e-03,2002-01-07,NaN
8,AB32,2002-01-09,57.015707,248.75,-2.320464e-13,-6.295724e-15,-7.901189e-15,-1.366566e-17,-3.817968e-18,-2.509400e-16,-8.347699e-15,-3.687507e-15,-2.531511e-16,0.001085,1.972815e-02,0.000544,6.826628e-04,1.226613e-03,2002-01-08,NaN
9,AB32,2002-01-10,57.015707,248.75,-4.922058e-15,-3.858655e-17,-9.167616e-17,-3.179468e-19,-1.419141e-19,-8.516404e-18,-4.706183e-16,-6.118856e-16,-1.252620e-17,0.000095,4.184641e-04,0.000003,7.920820e-06,1.125470e-05,2002-01-09,NaN


In [5]:
#Read in files above and confirm that samplingInt worked for the correct sites
pathData = '/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/Daily_Timeseries_NADPsites/'
os.chdir(pathData)
df1 = pd.read_csv('SO4_NO3_NH3_NH4_wetdep_Daily_SamplingIntAssigned_Allsites.csv')
df1
#df2 = pd.read_csv('TMQ_SO4_NO3_NH4_wetdep_Daily_SamplingIntAssigned_sitesIndex171_339.csv')

#Combine two iterations of sampling interval assignments
#combined_df = df1.combine_first(df2)
#combined_df = combined_df.loc[:,~combined_df.columns.duplicated()] # Drop duplicate columns
#combined_df.head()

#Confirm that this worked
#sampling_int_present = combined_df.groupby('siteId')['SamplingInt'].apply(lambda x: not x.isnull().all())
#sampling_int_absent = sampling_int_present[~sampling_int_present]
#print(sampling_int_absent) # all of these sites are discontinued prior to 2001 start date

#Filter the combined df for only sites that have a matching record
#filtered_df = combined_df[~combined_df['siteId'].isin(sampling_int_absent.index)]
#sampling_int_present = filtered_df.groupby('siteId')['SamplingInt'].apply(lambda x: not x.isnull().all())
#sampling_int_absent = sampling_int_present[~sampling_int_present]
#print(sampling_int_absent) # this should be empty (and it is!)

,Unnamed: 0,siteId,model_time,lat,lon,WD_HNO3,WD_NH4,WD_NH3,so4_a1SFWET,so4_a2SFWET,...,so4_c1SFWET,so4_c2SFWET,so4_c3SFWET,Mod_SO4_mgm2,Mod_NO3_mgm2,Mod_NH4_mgm2,Mod_NH3_mgm2,Mod_NH4_NH3_mgm2,time,SamplingInt
0,0,AB32,2002-01-01,57.015707,248.75,-1.395922e-25,1.784353e-27,-9.164005e-31,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,1.186787e-14,0.000000,7.917701e-20,7.917701e-20,2001-12-31,NaN
1,1,AB32,2002-01-02,57.015707,248.75,-1.970314e-15,-2.079935e-17,-1.916300e-17,0.000000e+00,0.000000e+00,...,-1.651060e-15,-1.473558e-17,-1.117747e-18,0.000144,1.675124e-04,0.000002,1.655683e-06,3.452747e-06,2002-01-01,NaN
2,2,AB32,2002-01-03,57.015707,248.75,-7.023518e-13,-4.363956e-14,-1.561877e-14,-2.656719e-16,-4.143680e-17,...,-3.057411e-13,-1.495660e-13,-7.769978e-14,0.046220,5.971262e-02,0.003770,1.349461e-03,5.119919e-03,2002-01-02,NaN
3,3,AB32,2002-01-04,57.015707,248.75,-7.138383e-12,-1.705666e-13,-6.441399e-13,-4.020032e-16,-5.714549e-17,...,-1.318216e-12,-1.442421e-13,-1.397894e-14,0.127930,6.068919e-01,0.014737,5.565369e-02,7.039064e-02,2002-01-03,NaN
4,4,AB32,2002-01-05,57.015707,248.75,-1.088934e-11,-1.722836e-12,-3.988795e-13,-7.043801e-16,-1.617886e-17,...,-1.701545e-12,-1.126615e-13,-2.767494e-15,0.157171,9.257913e-01,0.148853,3.446319e-02,1.833162e-01,2002-01-04,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2999356,2999356,WY96,2022-12-28,40.994764,253.75,-7.116247e-13,-1.894281e-14,-6.324074e-14,-8.389550e-17,-2.132509e-17,...,-2.665699e-13,-1.207700e-15,-4.008116e-14,0.026689,6.050099e-02,0.001637,5.464000e-03,7.100659e-03,2022-12-27,NaN
2999357,2999357,WY96,2022-12-29,40.994764,253.75,-7.098700e-12,-7.136778e-13,-1.430007e-12,-8.317375e-17,-3.172097e-17,...,-2.111419e-12,-2.635072e-13,-2.606395e-14,0.207549,6.035181e-01,0.061662,1.235526e-01,1.852144e-01,2022-12-28,NaN
2999358,2999358,WY96,2022-12-30,40.994764,253.75,-1.506921e-12,-8.867977e-14,-1.188008e-12,-2.433409e-18,-5.659553e-19,...,-9.677841e-13,-1.519494e-13,-2.134975e-15,0.096934,1.281156e-01,0.007662,1.026439e-01,1.103058e-01,2022-12-29,NaN
2999359,2999359,WY96,2022-12-31,40.994764,253.75,-7.532679e-12,-2.702265e-13,-1.359098e-12,-3.263604e-18,-9.396293e-19,...,-6.739652e-13,-2.778739e-14,-1.925901e-14,0.062308,6.404142e-01,0.023348,1.174261e-01,1.407736e-01,2022-12-30,NaN


In [6]:
#Read in the new precip files
precip_result_dir = Path('/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/Timeseries_atm_h2/')
precip_files_to_open = ['FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.PRECC.nc',
                           'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.PRECL.nc']

# Concatenate directory path with each file name separately
file_paths = [precip_result_dir / file_name for file_name in precip_files_to_open]

# Open multiple netCDF files as a single dataset
nc_precip = xr.open_mfdataset(file_paths, combine='nested') # units are in m/s
nc_precip['PRECC_mm'] = nc_precip['PRECC']* (86400*1000) # seconds to days and m to mm = mm/d
nc_precip['PRECL_mm'] = nc_precip['PRECL']* (86400*1000)
nc_precip['PREC_tot_mm'] = nc_precip['PRECC_mm']  + nc_precip['PRECL_mm']
nc_precip

<xarray.Dataset>
Dimensions:      (time: 7671, lat: 192, lon: 288)
Coordinates:
  * lat          (lat) float64 -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon          (lon) float64 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.2 357.5 358.8
  * time         (time) datetime64[ns] 2002-01-01 2002-01-02 ... 2023-01-01
Data variables:
    PRECC        (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    PRECL        (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    PRECC_mm     (time, lat, lon) float64 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    PRECL_mm     (time, lat, lon) float64 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    PREC_tot_mm  (time, lat, lon) float64 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
Attributes:
    Conventions:       CF-1.0
    source:            CAM
    case:              FCnudged_f09.mam.Murray.Apr11_01.2002_2023.001
    logname:           demurray
    host:              derecho1
    initial_file:      /glade/campaign/acom/acom-climate/UTLS/shawnh/archive/...
    topography_file:   /glade/campaign/cesm/cesmdata/inputdata/atm/cam/topo/f...
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    time_period_freq:  day_1

In [7]:
#Filter precip values for NADP sites by lat/lon
pathData = '/glade/u/home/demurray/External File Uploads/NADP NTN wet dep data/'
os.chdir(pathData)
ntn = pd.read_csv('ntn.csv')

# Only select the cells in .nc that correspond to an NADP site lat/long
subset_list = []
progress_bar = tqdm(total=len(ntn))
for index, row in ntn.iterrows():
    lat = row['latitude']
    lon = 360-(row['longitude']*-1)   # longitude is positive and based on 360 degrees.
    lat_idx = find_index(nc_precip['lat'].values, lat)   # right now we are doing a 'find nearest' calculation, should probably interpolate across grid cell and have exact coordinates represented?
    lon_idx = find_index(nc_precip['lon'].values, lon)
    subset = nc_precip.isel(lat=lat_idx, lon=lon_idx)
    subset['siteId'] = row['siteId']  # Add 'siteId' as a new coordinate/index
    subset = subset.assign_coords(siteId=row['siteId'])
    subset_list.append(subset)
    progress_bar.update(1)
progress_bar.close()

# Concatenate the list of subsets into a new xarray dataset
nc_precip_nadp = xr.concat(subset_list, dim='siteId')
nc_precip_nadp

100%|██████████| 391/391 [00:00<00:00, 433.33it/s]


<xarray.Dataset>
Dimensions:      (siteId: 391, time: 7671)
Coordinates:
    lat          (siteId) float64 57.02 56.07 57.02 65.5 ... 42.88 39.11 40.99
    lon          (siteId) float64 248.8 248.8 248.8 212.5 ... 271.2 280.0 253.8
  * time         (time) datetime64[ns] 2002-01-01 2002-01-02 ... 2023-01-01
  * siteId       (siteId) <U4 'AB32' 'AB34' 'AB36' ... 'WI99' 'WV99' 'WY96'
Data variables:
    PRECC        (siteId, time) float32 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    PRECL        (siteId, time) float32 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    PRECC_mm     (siteId, time) float64 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    PRECL_mm     (siteId, time) float64 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    PREC_tot_mm  (siteId, time) float64 dask.array<chunksize=(1, 7671), meta=np.ndarray>
Attributes:
    Conventions:       CF-1.0
    source:            CAM
    case:              FCnudged_f09.mam.Murray.Apr11_01.2002_2023.001
    logname:           demurray
    host:              derecho1
    initial_file:      /glade/campaign/acom/acom-climate/UTLS/shawnh/archive/...
    topography_file:   /glade/campaign/cesm/cesmdata/inputdata/atm/cam/topo/f...
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    time_period_freq:  day_1

In [8]:
#Convert to dataframe and assign new time
mod_nadp_precip = nc_precip_nadp.to_dataframe().reset_index()
mod_nadp_precip = mod_nadp_precip.rename(columns = {'time': 'model_time'})

#IMPORTANT STEP: change the time to one day prior (model writes time at end of current day)
mod_nadp_precip['time'] = pd.to_datetime(mod_nadp_precip['model_time'],format='%Y-%m-%d')+DateOffset(days=-1)
mod_nadp_precip = mod_nadp_precip[['siteId', 'time', 'PREC_tot_mm', 'PRECC_mm', 'PRECL_mm']]
mod_nadp_precip.head(10)

,siteId,time,PREC_tot_mm,PRECC_mm,PRECL_mm
0,AB32,2001-12-31,3.187350e-13,0.0,3.187350e-13
1,AB32,2002-01-01,2.667403e-05,0.0,2.667403e-05
2,AB32,2002-01-02,4.805037e-01,0.0,4.805037e-01
3,AB32,2002-01-03,1.796528e+00,0.0,1.796528e+00
4,AB32,2002-01-04,9.975159e-01,0.0,9.975159e-01
5,AB32,2002-01-05,1.829561e-01,0.0,1.829561e-01
6,AB32,2002-01-06,1.969842e-01,0.0,1.969842e-01
7,AB32,2002-01-07,1.154322e+00,0.0,1.154322e+00
8,AB32,2002-01-08,5.340185e-02,0.0,5.340185e-02
9,AB32,2002-01-09,4.679861e-04,0.0,4.679861e-04


In [9]:
#IGNORE THIS CELL IF RUNNING FULL ANALYSES, THIS IS JUST FOR COMPARISON WITH MUSICAV0 RUN
#truncate time
start = dt.datetime.strptime('2016-12-31', '%Y-%m-%d')
end = dt.datetime.strptime('2018-12-31', '%Y-%m-%d')
mod_nadp_precip_sub = mod_nadp_precip.loc[(mod_nadp_precip.time > start) & (mod_nadp_precip.time < end),]

#truncate variables
mod_nadp_precip_sub = mod_nadp_precip_sub[['siteId', 'time', 'PREC_tot_mm']]

#write file
mod_nadp_precip_sub.to_csv('/glade/u/home/demurray/Murray-NCAR-GVP/MUSICA Analyses/Precip_global20172018_cleaned_nadpsitemerged.csv')

In [11]:
#Read in the other df with wet dep variables, drop TMQ, 
pathData = '/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/Daily_Timeseries_NADPsites/'
os.chdir(pathData)
mod_nadp_chem = pd.read_csv('SO4_NO3_NH3_NH4_wetdep_Daily_SamplingIntAssigned_Allsites.csv', parse_dates = ['time', 'model_time'])
mod_nadp_chem = mod_nadp_chem[['siteId', 'model_time','time', 'SamplingInt', 'lat', 'lon', 'WD_HNO3', 'WD_NH4', 'WD_NH3', 'so4_a1SFWET', 'so4_a2SFWET', 'so4_a3SFWET',
       'so4_c1SFWET', 'so4_c2SFWET', 'so4_c3SFWET', 'Mod_SO4_mgm2', 'Mod_NO3_mgm2', 'Mod_NH4_mgm2', 'Mod_NH3_mgm2', 'Mod_NH4_NH3_mgm2']]
#mod_nadp_chem.columns

#Merge with precip df on siteId and time
mod_nadp_all = pd.merge(mod_nadp_chem, mod_nadp_precip, on = ['siteId', 'time'])
mod_nadp_all.to_csv('/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/Daily_Timeseries_NADPsites/PREC_SO4_NO3_NH4_NH3_wetdep_Daily_SamplingIntAssigned_Allsites.csv')
mod_nadp_all

,siteId,model_time,time,SamplingInt,lat,lon,WD_HNO3,WD_NH4,WD_NH3,so4_a1SFWET,...,so4_c2SFWET,so4_c3SFWET,Mod_SO4_mgm2,Mod_NO3_mgm2,Mod_NH4_mgm2,Mod_NH3_mgm2,Mod_NH4_NH3_mgm2,PREC_tot_mm,PRECC_mm,PRECL_mm
0,AB32,2002-01-01,2001-12-31,NaN,57.015707,248.75,-1.395922e-25,1.784353e-27,-9.164005e-31,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000,1.186787e-14,0.000000,7.917701e-20,7.917701e-20,3.187350e-13,0.0,3.187350e-13
1,AB32,2002-01-02,2002-01-01,NaN,57.015707,248.75,-1.970314e-15,-2.079935e-17,-1.916300e-17,0.000000e+00,...,-1.473558e-17,-1.117747e-18,0.000144,1.675124e-04,0.000002,1.655683e-06,3.452747e-06,2.667403e-05,0.0,2.667403e-05
2,AB32,2002-01-03,2002-01-02,NaN,57.015707,248.75,-7.023518e-13,-4.363956e-14,-1.561877e-14,-2.656719e-16,...,-1.495660e-13,-7.769978e-14,0.046220,5.971262e-02,0.003770,1.349461e-03,5.119919e-03,4.805037e-01,0.0,4.805037e-01
3,AB32,2002-01-04,2002-01-03,NaN,57.015707,248.75,-7.138383e-12,-1.705666e-13,-6.441399e-13,-4.020032e-16,...,-1.442421e-13,-1.397894e-14,0.127930,6.068919e-01,0.014737,5.565369e-02,7.039064e-02,1.796528e+00,0.0,1.796528e+00
4,AB32,2002-01-05,2002-01-04,NaN,57.015707,248.75,-1.088934e-11,-1.722836e-12,-3.988795e-13,-7.043801e-16,...,-1.126615e-13,-2.767494e-15,0.157171,9.257913e-01,0.148853,3.446319e-02,1.833162e-01,9.975159e-01,0.0,9.975159e-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2999356,WY96,2022-12-28,2022-12-27,NaN,40.994764,253.75,-7.116247e-13,-1.894281e-14,-6.324074e-14,-8.389550e-17,...,-1.207700e-15,-4.008116e-14,0.026689,6.050099e-02,0.001637,5.464000e-03,7.100659e-03,9.918514e-01,0.0,9.918514e-01
2999357,WY96,2022-12-29,2022-12-28,NaN,40.994764,253.75,-7.098700e-12,-7.136778e-13,-1.430007e-12,-8.317375e-17,...,-2.635072e-13,-2.606395e-14,0.207549,6.035181e-01,0.061662,1.235526e-01,1.852144e-01,3.069196e+00,0.0,3.069196e+00
2999358,WY96,2022-12-30,2022-12-29,NaN,40.994764,253.75,-1.506921e-12,-8.867977e-14,-1.188008e-12,-2.433409e-18,...,-1.519494e-13,-2.134975e-15,0.096934,1.281156e-01,0.007662,1.026439e-01,1.103058e-01,9.923106e-02,0.0,9.923106e-02
2999359,WY96,2022-12-31,2022-12-30,NaN,40.994764,253.75,-7.532679e-12,-2.702265e-13,-1.359098e-12,-3.263604e-18,...,-2.778739e-14,-1.925901e-14,0.062308,6.404142e-01,0.023348,1.174261e-01,1.407736e-01,7.748705e-02,0.0,7.748705e-02


In [12]:
mod_nadp_all.columns

Index(['siteId', 'model_time', 'time', 'SamplingInt', 'lat', 'lon', 'WD_HNO3',
       'WD_NH4', 'WD_NH3', 'so4_a1SFWET', 'so4_a2SFWET', 'so4_a3SFWET',
       'so4_c1SFWET', 'so4_c2SFWET', 'so4_c3SFWET', 'Mod_SO4_mgm2',
       'Mod_NO3_mgm2', 'Mod_NH4_mgm2', 'Mod_NH3_mgm2', 'Mod_NH4_NH3_mgm2',
       'PREC_tot_mm', 'PRECC_mm', 'PRECL_mm'],
      dtype='object')

In [13]:
#Then group by siteId and sampling int and sum variables below.
mod_nadp_sum = mod_nadp_all.groupby(['siteId', 'SamplingInt', 'lat', 'lon'])[['PREC_tot_mm', 'PRECC_mm', 'PRECL_mm', 'Mod_SO4_mgm2', 'Mod_NO3_mgm2','Mod_NH4_mgm2', 'Mod_NH3_mgm2', 'Mod_NH4_NH3_mgm2']].sum().reset_index()
mod_nadp_sum

mod_nadp_sum = pd.merge(mod_nadp_sum, nadp_df, how = 'left', on = ['siteId', 'SamplingInt'])
mod_nadp_sum.to_csv('/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/SamplingInt_Timeseries_NADPsites/PREC_SO4_NO3_NH4_NH3_Timeseries.SamplingInt_Summed_pairedNADPsites.csv')
mod_nadp_sum.head(20)

,siteId,SamplingInt,lat,lon,PREC_tot_mm,PRECC_mm,PRECL_mm,Mod_SO4_mgm2,Mod_NO3_mgm2,Mod_NH4_mgm2,...,longitude,dateOn,dateOff,dateOnround,dateOffround,SO4_mgm2,NO3_mgm2,NH4_mgm2,ppt_mm,IntTime
0,AB32,0.0,57.015707,248.75,22.086179,2.128905,19.957274,3.177832,1.855587,2.294607e-01,...,-111.6406,2016-09-13 18:40:00,2016-09-20 15:10:00,2016-09-14,2016-09-21,NaN,NaN,NaN,0.762,6 days 20:30:00
1,AB32,1.0,57.015707,248.75,3.994071,0.000000,3.994071,1.479748,1.487003,1.880133e-01,...,-111.6406,2016-09-20 15:15:00,2016-09-28 16:00:00,2016-09-21,2016-09-29,NaN,NaN,NaN,0.508,8 days 00:45:00
2,AB32,2.0,57.015707,248.75,23.925135,0.000000,23.925135,8.839498,11.882379,4.646642e+00,...,-111.6406,2016-09-28 16:00:00,2016-10-05 16:55:00,2016-09-29,2016-10-06,6.888988,3.426460,0.703326,18.034,7 days 00:55:00
3,AB32,3.0,57.015707,248.75,4.555199,0.000000,4.555199,0.423942,1.070503,1.240417e-01,...,-111.6406,2016-10-05 16:55:00,2016-10-11 17:00:00,2016-10-06,2016-10-12,NaN,NaN,NaN,10.160,6 days 00:05:00
4,AB32,4.0,57.015707,248.75,18.697038,0.000000,18.697038,5.312534,8.021285,1.206460e+00,...,-111.6406,2016-10-11 17:00:00,2016-10-18 20:00:00,2016-10-12,2016-10-19,NaN,NaN,NaN,13.208,7 days 03:00:00
5,AB32,5.0,57.015707,248.75,8.727634,0.000000,8.727634,2.671013,6.907131,7.867468e-01,...,-111.6406,2016-10-18 20:00:00,2016-10-25 18:00:00,2016-10-19,2016-10-26,3.940048,6.443472,1.219708,3.556,6 days 22:00:00
6,AB32,6.0,57.015707,248.75,9.459058,0.000000,9.459058,2.374204,7.741049,1.769656e+00,...,-111.6406,2016-10-25 18:00:00,2016-10-31 16:47:00,2016-10-26,2016-11-01,6.391656,9.125712,3.959352,4.572,5 days 22:47:00
7,AB32,7.0,57.015707,248.75,1.986510,0.000000,1.986510,0.421055,0.630983,9.124981e-02,...,-111.6406,2016-10-31 16:47:00,2016-11-07 17:00:00,2016-11-01,2016-11-08,0.890016,0.431292,0.190500,1.524,7 days 00:13:00
8,AB32,8.0,57.015707,248.75,4.096038,0.000000,4.096038,0.313753,1.084822,6.615716e-02,...,-111.6406,2016-11-07 17:00:00,2016-11-15 16:00:00,2016-11-08,2016-11-16,0.141732,0.358902,0.068326,0.254,7 days 23:00:00
9,AB32,9.0,57.015707,248.75,4.560259,0.000000,4.560259,0.711487,6.759315,3.382609e-01,...,-111.6406,2016-11-15 16:00:00,2016-11-22 18:30:00,2016-11-16,2016-11-23,0.672084,1.280160,0.083312,0.508,7 days 02:30:00
